# Module 3.1: Linear & Regularized Regression for Plant Energy Forecasting
### 🇵🇱 Regresja Liniowa i Regularyzacja w Prognozowaniu Zużycia Energii

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/max480425/industrial-ml-course/blob/main/public/notebooks/m03_l01_linear_regularized_regression_energy.ipynb)

**Industrial Machine Learning & Predictive Maintenance Interactive Laboratory**

> **Summary:** Forecast factory electric kilowatt load using Ridge (L2) and Lasso (L1) regression, handling multicollinear sensor tags and physical constraints.  
> 🇵🇱 **Streszczenie:** Prognozuj zapotrzebowanie fabryki na moc (kW) za pomocą regresji Ridge i Lasso, opanowując współliniowość czujników i ograniczenia fizyczne.

- **Estimated Time:** 25 minutes
- **Runtime:** Python 3.10+ (CPU or Free Tier T4 GPU)
- **Packages:** `numpy`, `scipy`, `scikit-learn`, `pandas`, `matplotlib`, `seaborn`
---


In [ ]:
# Step 1: Install & Verify Python Scientific Environment
!pip install -q numpy scipy scikit-learn pandas matplotlib seaborn

import numpy as np
import pandas as pd
import scipy.signal as signal
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.family'] = 'monospace'
print("✓ Industrial ML environment loaded successfully.")


### Key Engineering Concepts:
1. **Industrial Energy Prediction**: Forecasting furnace power (MWh) based on production tonnage and thermocouple readings.
2. **Severe Multicollinearity**: In an industrial furnace, 10+ thermocouples placed along a 30-meter tunnel correlate at $r > 0.95$. Ordinary Least Squares (OLS) produces wild, unstable coefficient swings.
3. **Ridge ($L_2$) vs Lasso ($L_1$)**:
   - **Ridge**: Shrinks correlated coefficients together, stabilizing variance.
   - **Lasso**: Drives non-informative thermocouple weights to exact zero, performing automatic sensor pruning.


In [ ]:
# 1. Simulate Industrial Furnace Energy Dataset with Collinear Sensors
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_squared_error, r2_score

np.random.seed(42)
n_runs = 800

# Primary operational drivers
production_tonnage = np.random.uniform(20.0, 100.0, n_runs)
ambient_temp = np.random.uniform(-5.0, 35.0, n_runs)

# 8 highly correlated zone thermocouples
core_heat = 500.0 + 3.5 * production_tonnage - 0.8 * ambient_temp
tc_sensors = np.column_stack([
    core_heat + np.random.normal(0, 4.0, n_runs) for _ in range(8)
])

X = np.column_stack([production_tonnage, ambient_temp, tc_sensors])
feature_names = ["tonnage", "ambient_temp"] + [f"tc_zone_{i+1}" for i in range(8)]

# True energy consumption [MWh]
true_energy = 15.0 + 0.45 * production_tonnage - 0.12 * ambient_temp + 0.02 * tc_sensors[:, 0] + np.random.normal(0, 1.2, n_runs)
print(f"✓ Generated furnace energy dataset: {n_runs} runs with {X.shape[1]} features.")


In [ ]:
# 2. Fit OLS vs Ridge vs Lasso & Plot Coefficients
ols = LinearRegression().fit(X, true_energy)
ridge = Ridge(alpha=100.0).fit(X, true_energy)
lasso = Lasso(alpha=0.5).fit(X, true_energy)

plt.figure(figsize=(12, 5))
x_idx = np.arange(len(feature_names))
width = 0.25

plt.bar(x_idx - width, ols.coef_, width=width, color='#ef4444', label=f'OLS (Unstable) R²={r2_score(true_energy, ols.predict(X)):.3f}')
plt.bar(x_idx, ridge.coef_, width=width, color='#0284c7', label=f'Ridge (L2 Shrinkage) R²={r2_score(true_energy, ridge.predict(X)):.3f}')
plt.bar(x_idx + width, lasso.coef_, width=width, color='#10b981', label=f'Lasso (L1 Pruning) R²={r2_score(true_energy, lasso.predict(X)):.3f}')

plt.xticks(x_idx, feature_names, rotation=35, ha='right')
plt.title("Regularized Regression: Taming Sensor Multicollinearity", fontsize=13, fontweight='bold')
plt.ylabel("Learned Feature Weight")
plt.legend()
plt.grid(True, linestyle='--', alpha=0.3)
plt.tight_layout()
plt.show()

# Count pruned features in Lasso
zero_weights = (lasso.coef_ == 0.0).sum()
print(f"Lasso automatically pruned {zero_weights} redundant thermocouple features to exact zero!")


## 📋 Practical Hands-On Checklist

- [ ] **Task 1:** Fit Ordinary Least Squares (OLS) regression on plant power consumption telemetry *(🇵🇱 Dopasuj klasyczną regresję OLS do danych poboru mocy fabryki)*
- [ ] **Task 2:** Apply Ridge (L2) regression to stabilize weights under high sensor multicollinearity *(🇵🇱 Zastosuj regularyzację Ridge (L2), stabilizując wagi przy skorelowanych czujnikach)*
- [ ] **Task 3:** Perform automated feature selection using Lasso (L1) sparsity penalty *(🇵🇱 Wykonaj selekcję kluczowych parametrów za pomocą rzadkich wag Lasso (L1))*


In [ ]:
# Exercise Challenge:
# Assert that Ridge regression achieves an R² score exceeding 0.95
r2_ridge = r2_score(true_energy, ridge.predict(X))
print(f"Ridge Model R² Score: {r2_ridge:.4f}")
assert r2_ridge > 0.95, "Ridge model failed to reach target accuracy."
print("✓ Regularized regression exercise passed.")
